# GNN with a matched graph density, and a fair "no graph" control

**Problem with the earlier GNN runs.** The train graph held all 1.63M training flows, while the test graph holds only the 0.41M test flows. Train endpoints therefore had about 4x more neighbours than test endpoints (the share of nodes with a single flow was 43% in train and 78% in test). The model was trained on a denser graph than it was tested on, which could hide a benefit of the graph.

**Fix.** The train graph is split into random chunks the size of the test set (4 chunks). Each chunk is its own graph: a node's neighbours come only from its own chunk, so train and test neighbourhoods have the same density. All chunks are stored as one graph made of disconnected pieces, so the training code is unchanged.

| Condition | What it shows |
|---|---|
| **no graph** | the same network with the node embeddings zeroed (flow features only). Does not depend on density |
| **real graph, full train graph** | the earlier setting (denser than test) |
| **real graph, matched density** | the fix |
| **shuffled edges, matched density** | each flow gets another flow's endpoints; same node degrees, no real connections |

5 seeds per condition (seed changes validation split, initialisation, chunk assignment and neighbour sampling; the same seeds for every condition). Test predictions are saved for the bootstrap analysis. Test data is never used for training or early stopping.

In [ ]:
import copy
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

DEBUG = False   # True = quick smoke test on a small random sample
SEEDS = [42] if DEBUG else [42, 43, 44, 45, 46]
DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN = 64
BATCH = 2048
MAX_EPOCHS = 2 if DEBUG else 12
PATIENCE = 3
LR = 2e-3
print("device:", DEVICE, "| debug:", DEBUG, "| seeds:", SEEDS)

## 1. Load data and build graphs

Nodes are (IP, port) endpoints and edges are flows, as in the GNN notebook. `chunk` (optional) makes the node key chunk-specific, so chunks do not share nodes.

In [ ]:
train_df = pd.read_parquet(f"{DATASET_PATH}/train.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")
if DEBUG:
    train_df, test_df = train_df.sample(150_000, random_state=0), test_df.sample(30_000, random_state=0)
train_graph = pd.read_csv(f"{DATASET_PATH}/train_graph.csv", index_col="row_id").loc[train_df.index]
test_graph = pd.read_csv(f"{DATASET_PATH}/test_graph.csv", index_col="row_id").loc[test_df.index]

class_names = joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_
n_classes = len(class_names)
feature_cols = [c for c in train_df.columns if c != "label"]
N_CHUNKS = max(1, round(len(train_df) / len(test_df)))
print("train edges", len(train_df), "| test edges", len(test_df), "| features", len(feature_cols), "| chunks", N_CHUNKS)


def build_graph(graph_df, feat_df, chunk=None):
    n = len(graph_df)
    ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ip_code = ip_code.astype("int64")
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    key = ip_code * 70_000 + ports
    if chunk is not None:                                   # separate node set per chunk
        key = key + np.concatenate([chunk, chunk]).astype("int64") * 10_000_000
    node_id, node_keys = pd.factorize(key)                  # one node per distinct (chunk, IP, port)
    src, dst = torch.from_numpy(node_id[:n]), torch.from_numpy(node_id[n:])
    n_nodes = len(node_keys)

    nodes = torch.cat([src, dst])          # every edge appears once per endpoint
    other = torch.cat([dst, src])
    edge_ids = torch.cat([torch.arange(n), torch.arange(n)])
    order = torch.argsort(nodes)
    ptr = torch.zeros(n_nodes + 1, dtype=torch.long)
    ptr[1:] = torch.cumsum(torch.bincount(nodes, minlength=n_nodes), 0)
    G = {"n_nodes": n_nodes, "src": src, "dst": dst, "ptr": ptr,
         "inc_edge": edge_ids[order], "inc_other": other[order],
         "x": torch.from_numpy(feat_df[feature_cols].to_numpy("float32")),
         "y": torch.from_numpy(feat_df["label"].to_numpy("int64"))}
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in G.items()}


def shuffle_endpoints(graph_df, seed):
    """Give every flow another flow's endpoints (the four columns move together)."""
    perm = np.random.default_rng(seed).permutation(len(graph_df))
    out = graph_df.copy()
    for col in ["srcip", "dstip", "sport", "dsport"]:
        out[col] = graph_df[col].to_numpy()[perm]
    return out


def sample_neighbors(G, nodes, fanout):
    """Draw `fanout` incident edges (with replacement) for each node -> (edge ids, neighbour node ids)."""
    start = G["ptr"][nodes]
    deg = G["ptr"][nodes + 1] - start
    offset = (torch.rand(len(nodes), fanout, device=nodes.device) * deg.unsqueeze(1)).long()
    offset = torch.minimum(offset, deg.unsqueeze(1) - 1)
    pos = start.unsqueeze(1) + offset
    return G["inc_edge"][pos], G["inc_other"][pos]


def chunks_for(seed):
    return np.random.default_rng(seed).integers(0, N_CHUNKS, len(train_df))


G_train_full = build_graph(train_graph, train_df)
G_train_chunk = build_graph(train_graph, train_df, chunk=chunks_for(42))
G_test = build_graph(test_graph, test_df)

## 2. Does the matched graph really have the test graph's density?

Share of nodes with a single flow, and mean flows per node, for the three graphs.

In [ ]:
density = {}
for name, G in [("train, full graph", G_train_full), ("train, matched chunks", G_train_chunk), ("test graph", G_test)]:
    deg = (G["ptr"][1:] - G["ptr"][:-1]).cpu().numpy()
    density[name] = {"nodes": len(deg), "edges": len(G["src"]), "degree-1 nodes": f"{(deg == 1).mean():.1%}",
                     "mean degree": round(float(deg.mean()), 2), "median degree": int(np.median(deg))}
print(pd.DataFrame(density).T.to_string())

## 3. Model

The same E-GraphSAGE as before (hidden 64, neighbours sampled 8 and 5, constant initial node feature, classifier on [source embedding, destination embedding, flow features]). `use_graph=False` zeroes both node embeddings.

In [ ]:
class EdgeSAGE(nn.Module):
    def __init__(self, n_feat, hidden, n_classes, fanout_2hop=8, fanout_1hop=5, use_graph=True):
        super().__init__()
        self.fanout_2hop, self.fanout_1hop, self.use_graph = fanout_2hop, fanout_1hop, use_graph
        self.edge1 = nn.Linear(n_feat, hidden)
        self.upd1 = nn.Linear(1 + hidden, hidden)
        self.node2 = nn.Linear(hidden, hidden, bias=False)
        self.edge2 = nn.Linear(n_feat, hidden)
        self.upd2 = nn.Linear(2 * hidden, hidden)
        self.head = nn.Sequential(nn.Linear(2 * hidden + n_feat, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, n_classes))

    def layer1(self, G, nodes):
        eids, _ = sample_neighbors(G, nodes, self.fanout_1hop)
        msg = torch.relu(self.edge1(G["x"][eids]))
        ones = torch.ones(len(nodes), 1, device=nodes.device)
        return torch.relu(self.upd1(torch.cat([ones, msg.mean(1)], dim=1)))

    def forward(self, G, idx):
        if not self.use_graph:
            zeros = torch.zeros(len(idx), self.head[0].in_features - G["x"].shape[1], device=idx.device)
            return self.head(torch.cat([zeros, G["x"][idx]], dim=1))
        ends = torch.cat([G["src"][idx], G["dst"][idx]])
        h1_self = self.layer1(G, ends)
        eids, nbrs = sample_neighbors(G, ends, self.fanout_2hop)
        h1_nbr = self.layer1(G, nbrs.reshape(-1)).view(len(ends), self.fanout_2hop, -1)
        msg = torch.relu(self.node2(h1_nbr) + self.edge2(G["x"][eids]))
        h2 = torch.relu(self.upd2(torch.cat([h1_self, msg.mean(1)], dim=1)))
        return self.head(torch.cat([h2[:len(idx)], h2[len(idx):], G["x"][idx]], dim=1))


@torch.no_grad()
def predict_proba(model, G, idx, passes=1, batch=8192):
    model.eval()
    out = []
    for i in range(0, len(idx), batch):
        b = idx[i:i + batch]
        out.append((sum(torch.softmax(model(G, b), 1) for _ in range(passes)) / passes).cpu())
    return torch.cat(out)

## 4. One run = one condition and one seed

15% of the train edges are held out for early stopping on validation macro F1 (their labels are hidden from the loss; their features sit in the graph, as before). Square-root balanced class weights, Adam. The test graph is used once at the end, with 5 neighbour-sampling passes.

In [ ]:
def run(name, seed, graph, use_graph=True, shuffle=False):
    t0 = time.time()
    torch.manual_seed(seed)
    np.random.seed(seed)
    if graph == "chunk":
        G_tr = build_graph(shuffle_endpoints(train_graph, seed) if shuffle else train_graph, train_df, chunk=chunks_for(seed))
    else:
        G_tr = G_train_full

    y_all = G_tr["y"].cpu().numpy()
    tr_np, va_np = train_test_split(np.arange(len(y_all)), test_size=0.15, stratify=y_all, random_state=seed)
    tr_idx, va_idx = torch.from_numpy(tr_np).to(DEVICE), torch.from_numpy(va_np).to(DEVICE)
    cw = np.sqrt(compute_class_weight("balanced", classes=np.arange(n_classes), y=y_all[tr_np]))
    class_weights = torch.tensor(cw, dtype=torch.float32, device=DEVICE)

    model = EdgeSAGE(len(feature_cols), HIDDEN, n_classes, use_graph=use_graph).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    y_val = G_tr["y"][va_idx].cpu().numpy()

    best, bad_epochs = {"f1": -1.0, "state": None, "epoch": 0}, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = tr_idx[torch.randperm(len(tr_idx), device=DEVICE)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad()
            loss_fn(model(G_tr, b), G_tr["y"][b]).backward()
            opt.step()
        val_f1 = f1_score(y_val, predict_proba(model, G_tr, va_idx).argmax(1).numpy(), average="macro")
        if val_f1 > best["f1"]:
            best, bad_epochs = {"f1": val_f1, "state": copy.deepcopy(model.state_dict()), "epoch": epoch}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    model.load_state_dict(best["state"])
    te_idx = torch.arange(len(G_test["src"]), device=DEVICE)
    pred = predict_proba(model, G_test, te_idx, passes=5).argmax(1).numpy()
    y_te = G_test["y"].cpu().numpy()
    per_class = f1_score(y_te, pred, labels=np.arange(n_classes), average=None, zero_division=0)
    print(f"{name:<36} seed {seed} | best epoch {best['epoch']:>2} | val F1 {best['f1']:.4f} | "
          f"test macro F1 {per_class.mean():.4f} | {time.time() - t0:.0f}s", flush=True)
    row = {"condition": name, "seed": seed, "val macro F1": best["f1"], "test macro F1": per_class.mean(),
           **{f"F1 {c}": f for c, f in zip(class_names, per_class)}}
    return row, pred.astype("int8")


CONDITIONS = {
    "no graph": dict(graph="full", use_graph=False),
    "real graph, full train graph": dict(graph="full"),
    "real graph, matched density": dict(graph="chunk"),
    "shuffled edges, matched density": dict(graph="chunk", shuffle=True),
}

## 5. Run all conditions

In [ ]:
rows, preds = [], {}
for seed in SEEDS:
    for name, kwargs in CONDITIONS.items():
        row, pred = run(name, seed, **kwargs)
        rows.append(row)
        preds[f"{name}__{seed}"] = pred
results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/gnn_density_runs.csv", index=False)
pd.DataFrame(preds, index=test_df.index).to_parquet("/kaggle/working/gnn_density_test_pred.parquet")

## 6. Results

Bars = mean over seeds, dots = individual seeds.

In [ ]:
summary = results.groupby("condition", sort=False)[["val macro F1", "test macro F1"]].agg(["mean", "std"])
print(summary.round(4).to_string())

order = list(CONDITIONS)
plt.figure(figsize=(10, 4))
sns.barplot(data=results, y="condition", x="test macro F1", order=order, errorbar="sd", capsize=0.15, color="steelblue", alpha=0.8)
sns.stripplot(data=results, y="condition", x="test macro F1", order=order, color="black", size=5)
plt.xlim(0.45, 0.7)
plt.title("GNN test macro F1 by condition (mean and std over seeds)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_01.png", dpi=130, bbox_inches="tight")
plt.show()

p = results.pivot(index="seed", columns="condition", values="test macro F1")
v = results.pivot(index="seed", columns="condition", values="val macro F1")
for a, b in [("real graph, matched density", "no graph"), ("real graph, matched density", "shuffled edges, matched density"),
             ("real graph, matched density", "real graph, full train graph"), ("real graph, full train graph", "no graph")]:
    dt, dv = p[a] - p[b], v[a] - v[b]
    print(f"{a} minus {b}\n   test per seed {dt.round(3).tolist()} | mean {dt.mean():+.4f}\n   val  per seed {dv.round(3).tolist()} | mean {dv.mean():+.4f}")